# Module 4 — Evaluation metrics (Telco churn): commented walkthrough

Alexey's lecture code (lessons 4.1 – 4.7), with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the machine-learning logic behind it (what could go wrong if we skipped it)
- `# PYTHON` how the Python / NumPy / pandas / scikit-learn syntax works, so you can write it yourself later

The original, uncommented notebook is untouched in `../notebooks/`.

Differences from the lecture:
- sections 4.1 – 4.7 each open with an **In plain words** cell. 4.2 – 4.6 also carry a tiny 10-customer toy example (`toy_` variables, numbers computed in code) so the idea is visible before it is applied to 1,409 real customers. The `toy_` prefix keeps them from overwriting the real `y_val`, `y_pred`, `tp`, `fp`, ...;
- the data is read from the local file (same `data-week-3.csv` as Module 3);
- text columns are found with `select_dtypes`. On pandas 3 the lecture's `df.dtypes == 'object'` selects **nothing**, so the values are never lower-cased, `churn == 'yes'` never matches, every target becomes 0 and `LogisticRegression` stops with *"only one class"*. The lecture notebook in `../notebooks/` crashes there for exactly this reason;
- the hand-typed numbers (`1132 / 1409`, the threshold `0.726`) are computed in code, so they match *your* run;
- `%autosave 0` and `!pip install tqdm` are dropped: the cross-validation loop just prints;
- the lecture stores its AUC in a variable called `auc`, which **overwrites** the `auc()` function imported in 4.6. Here the variable is called `score`;
- the lecture uses `LogisticRegression()` defaults. **HW4 pins** `solver='liblinear', C=1.0, max_iter=1000` and a different dataset, so HW4 numbers will differ from these. The split seeds (`random_state=1`) are the same as HW4's.

## 4.1 Evaluation metrics: session overview

A **metric** is a function: predictions in, true values in, one number out. Before we can measure anything we need
the Module 3 model back, so the first cells are a compact recap of 3.2 – 3.10.

### In plain words: what this module is about

The Module 3 model does not say "churn" or "stays". It gives each customer a **probability** (say 0.83). Everything here answers one question: **how good is that model, honestly?**

"Honestly" means measuring on customers the model has never seen (the validation set). A model can memorise its training data and look perfect there.

The sections build in order: 4.2 accuracy, and why it can mislead. 4.3 the four kinds of right and wrong. 4.4 two questions about the customers we flag (precision and recall). 4.5 – 4.6 a way to judge the model without choosing a threshold (ROC and AUC). 4.7 repeating the whole test several times to see how much the number wobbles (cross-validation).

In [1]:
# PYTHON: 'import X as Y' loads a library under a short nickname.
import pandas as pd                 # tables (DataFrame)
import numpy as np                  # arrays and vectorised maths: the metrics are mostly array comparisons
import matplotlib.pyplot as plt     # plots: accuracy vs threshold, TPR/FPR, ROC curve

# PYTHON: 'from module import name' brings in just that one name.
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline

In [3]:
# GOAL: load and clean the Telco data exactly as in Module 3.
df = pd.read_csv('data-week-3.csv')

# PYTHON: .str applies string methods to every column name at once; chaining feeds one result into the next.
df.columns = df.columns.str.lower().str.replace(' ', '_')

# WHY: select_dtypes works on old pandas (text = 'object') AND pandas 3 (text = 'str').
#      The lecture's df.dtypes == 'object' finds no columns on pandas 3 -> nothing gets cleaned below.
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# WHY: totalcharges is text because 11 brand-new customers have a blank value. errors='coerce' turns
#      those into NaN, and 0 is their true value (nothing billed yet). See the Module 3 walkthrough.
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

# PYTHON: (df.churn == 'yes') is a True/False column; .astype(int) turns True/False into 1/0.
df.churn = (df.churn == 'yes').astype(int)

# Sanity check: the churn rate must be ~0.27. If it prints 0.0, the cleaning above did not run.
df.churn.mean().round(3)

FileNotFoundError: [Errno 2] No such file or directory: 'data-week-3.csv'

In [ ]:
# GOAL: the same 60 / 20 / 20 split as Module 3 (and as HW4: random_state=1 both times).
# PYTHON: train_test_split shuffles the rows and cuts them in two. test_size=0.2 -> 20% goes to the second part.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
# WHY: 0.25 of the remaining 80% = 20% of the whole -> train 60%, validation 20%, test 20%.
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# PYTHON: reset_index(drop=True) renumbers the rows 0, 1, 2, ... and throws the old shuffled index away.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# PYTHON: .values turns a pandas column into a plain NumPy array (what sklearn and our comparisons expect).
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

# WHY: remove the target from the inputs, or the model simply reads the answer.
#      (df_full_train keeps its churn column on purpose: cross-validation in 4.7 needs it.)
del df_train['churn']
del df_val['churn']
del df_test['churn']

In [ ]:
# GOAL: name the features once so every later step uses exactly the same list.
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
    'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod',
]

In [ ]:
# GOAL: train the Module 3 model.
# WHY: DictVectorizer is FIT on train only (learns the one-hot column layout) and only TRANSFORMs validation.
dv = DictVectorizer(sparse=False)

# PYTHON: to_dict(orient='records') -> a list with one {column: value} dict per row (DictVectorizer's input format).
train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

# NOTE: you may see a ConvergenceWarning: totalcharges (thousands) and the 0/1 columns have very different
#       scales, so lbfgs needs many steps. The model is fine; scaling is the cure (Module 3 scaling notebook).
model = LogisticRegression()
model.fit(X_train, y_train)

/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
# GOAL: predict churn PROBABILITIES for validation, turn them into decisions, measure accuracy.
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

# PYTHON: predict_proba returns one column per class: [:, 0] = P(stay), [:, 1] = P(churn). [:, 1] takes column 1.
y_pred = model.predict_proba(X_val)[:, 1]

# WHY: the model outputs a probability; a DECISION needs a threshold. 0.5 is a convention, not a law.
churn_decision = (y_pred >= 0.5)

# PYTHON: comparing two arrays gives True/False per row; the mean of booleans = the fraction of True.
# This fraction of correct decisions is ACCURACY.
(y_val == churn_decision).mean()

## 4.2 Accuracy and dummy model

Accuracy = correct decisions / all decisions. Two questions: does 0.5 give the best accuracy, and how good is
~80% really? We answer the second with a **dummy model** that ignores the customer completely.

In [ ]:
# GOAL: accuracy written out as the fraction it is.
# PYTHON: .sum() of a boolean array counts the Trues.
n_correct = (y_val == churn_decision).sum()
n_total = len(y_val)

# The lecture types these two numbers in by hand (1132 / 1409); computing them keeps them right for any run.
n_correct, n_total, n_correct / n_total

In [ ]:
# GOAL: the same number with scikit-learn.
# PYTHON: every sklearn metric has the signature metric(y_true, y_pred_or_score). Order matters for some metrics.
from sklearn.metrics import accuracy_score

accuracy_score(y_val, y_pred >= 0.5)

### In plain words: the threshold is a dial

The model gives a probability, but to *act* (send a discount, or not) we need a yes/no. The rule is `probability >= threshold`.

Threshold 0.5 is the middle setting. Turn the dial down to 0.1 and almost everybody is flagged; turn it up to 0.9 and almost nobody is. 0.5 is a habit, not a law, so the next cell tries 21 settings and shows what each one gives. Everything up to 4.6 is about what happens as this dial moves.

In [ ]:
# GOAL: try 21 thresholds from 0 to 1 and see where accuracy peaks.
# PYTHON: np.linspace(start, stop, num) gives num evenly spaced values INCLUDING both ends: 0.00, 0.05, ..., 1.00.
thresholds = np.linspace(0, 1, 21)

scores = []

for t in thresholds:
    score = accuracy_score(y_val, y_pred >= t)
    # PYTHON: '%.2f %.3f' % (a, b) is old-style formatting: a with 2 decimals, b with 3.
    print('%.2f %.3f' % (t, score))
    scores.append(score)

In [ ]:
# WHY: the curve is flat around 0.5, so 0.5 was a fine choice. Look at the two ENDS of the curve next.
plt.plot(thresholds, scores)
plt.xlabel('threshold')
plt.ylabel('accuracy')

In [ ]:
# GOAL: what does threshold 1.0 mean? No probability is >= 1.0, so the model predicts "no churn" for EVERYONE.
# PYTHON: Counter counts how many times each value occurs (here: how many True and False).
from collections import Counter

Counter(y_pred >= 1.0)

In [ ]:
# GOAL: accuracy of that dummy "nobody churns" model.
# WHY: it is right for every non-churner, i.e. 1 - churn rate. Our real model is only a few points better.
#      This is CLASS IMBALANCE: when one class dominates, accuracy rewards predicting the majority class.
#      Accuracy alone cannot tell a useful churn model from one that never flags anybody.
1 - y_val.mean()

### In plain words: the dummy model

Picture a validation set of just 10 customers, 3 of whom churn. A "model" that never looks at anybody and says **stays** for all 10 is right for the 7 who stay and wrong for the 3 who leave: **70% accuracy**, and it catches **zero** churners.

So an accuracy number only means something next to this baseline. Our real model's ~80% is a gain of (80% − 73%), not "80% good".

The baseline is the share of the biggest class, so it depends on how unbalanced the data is. If only 5% of customers churned, "nobody churns" would be right 95% of the time, and a model with 90% accuracy would be *worse* than doing nothing. **Habit: compute the dummy's accuracy before you trust any accuracy.**

In [ ]:
# A toy validation set: 10 customers, 3 churners (1 = churn, 0 = stays).
# PYTHON: the 'toy_' prefix keeps these names from overwriting the real y_val / y_pred used later.
toy_y = np.array([0, 0, 1, 0, 0, 1, 0, 0, 1, 0])

# The dummy model: predict "stays" (0) for everybody.
toy_dummy = np.zeros(10, dtype=int)

# Its accuracy, two ways: compare-and-average, or 1 - churn rate. Same number.
(toy_y == toy_dummy).mean(), 1 - toy_y.mean()

## 4.3 Confusion table

Every decision is one of four cases. Being *right* splits into TP / TN, being *wrong* into FP / FN, and the two
kinds of error cost different things (a discount wasted on a loyal customer vs a churner we never contacted).

### In plain words: four outcomes

Every decision about a customer lands in one of four boxes. Read each name in two halves: the **second word** is what the model *said* (Positive = "will churn", Negative = "will stay"), the **first word** says whether it was *right* (True / False).

| | model said **churn** | model said **stays** |
|---|---|---|
| customer **did churn** | **TP** true positive: right, we can act | **FN** false negative: a churner we miss |
| customer **stayed** | **FP** false positive: a false alarm, a discount wasted | **TN** true negative: right, nothing to do |

The two mistakes (FP and FN) cost different things, and accuracy adds them together as if they were the same. Run the toy cell: the "model" flags 4 customers and gets the **same 70% accuracy as the do-nothing dummy**, yet it finds 2 of the 3 churners. The table shows the difference; accuracy cannot.

In [ ]:
toy_y    = np.array([0, 0, 1, 0, 0, 1, 0, 0, 1, 0])   # truth: 3 churners
toy_flag = np.array([0, 1, 1, 1, 0, 0, 0, 0, 1, 0])   # decision: the model flags 4 customers

# PYTHON: & is element-wise AND on arrays; .sum() counts the Trues.
toy_tp = ((toy_flag == 1) & (toy_y == 1)).sum()   # flagged and churned
toy_fp = ((toy_flag == 1) & (toy_y == 0)).sum()   # flagged but stayed   -> false alarm
toy_fn = ((toy_flag == 0) & (toy_y == 1)).sum()   # not flagged, churned -> missed churner
toy_tn = ((toy_flag == 0) & (toy_y == 0)).sum()   # not flagged, stayed

print('TP', toy_tp, ' FP', toy_fp, ' FN', toy_fn, ' TN', toy_tn)
print('accuracy', (toy_tp + toy_tn) / len(toy_y))

In [ ]:
# GOAL: two masks for the TRUTH.
# PYTHON: a comparison on an array gives a boolean array of the same length (a "mask").
actual_positive = (y_val == 1)
actual_negative = (y_val == 0)

In [ ]:
# GOAL: two masks for the DECISION at threshold t.
t = 0.5
predict_positive = (y_pred >= t)
predict_negative = (y_pred < t)

In [ ]:
# GOAL: count the four cells of the table.
# PYTHON: & is element-wise AND on boolean arrays (True only where BOTH are True). Use &, not 'and':
#         'and' works on single True/False values and raises an error on arrays.
tp = (predict_positive & actual_positive).sum()   # said churn,    did churn
tn = (predict_negative & actual_negative).sum()   # said stays,    did stay

fp = (predict_positive & actual_negative).sum()   # said churn,    stayed  -> false alarm
fn = (predict_negative & actual_positive).sum()   # said stays,    churned -> missed churner

tp, tn, fp, fn

In [ ]:
# GOAL: arrange the counts as a table: rows = truth (0, 1), columns = decision (0, 1).
# PYTHON: np.array([[...], [...]]) builds a 2-D array from a list of rows.
confusion_matrix = np.array([
    [tn, fp],
    [fn, tp]
])
confusion_matrix

In [ ]:
# GOAL: the same table as fractions of all customers.
# WHY: the diagonal (tn + tp) is the accuracy. Off the diagonal you can now SEE which error we make more often.
# PYTHON: dividing an array by a number divides every cell (broadcasting).
(confusion_matrix / confusion_matrix.sum()).round(2)

In [ ]:
# Cross-check with scikit-learn: same layout (rows = truth, columns = prediction), same numbers.
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

sk_confusion_matrix(y_val, y_pred >= 0.5)

## 4.4 Precision and recall

- **Precision** = TP / (TP + FP): of the customers we *flagged*, how many really churn? (quality of the flag)
- **Recall** = TP / (TP + FN): of the customers who *really churn*, how many did we flag? (coverage)

Both ignore TN, the big majority class, which is why they don't flatter us the way accuracy does.

### In plain words: two questions about two different groups

- **Precision** looks only at the customers we *flagged*: "we sent 4 discounts, how many went to someone really about to leave?" High precision = little money wasted.
- **Recall** looks only at the customers who *really churn*: "there were 3 real churners, how many did we reach?" High recall = few customers lost.

They pull against each other as the threshold moves. A **wide net** (low threshold) reaches nearly every churner (recall high) but also many loyal customers (precision low). A **narrow net** (high threshold) flags only the sure cases (precision high) and lets many churners slip by (recall low). Which side to favour is a business decision (what a discount costs vs what a lost customer costs), not a maths one.

The same toy numbers as in 4.3:

In [ ]:
toy_y    = np.array([0, 0, 1, 0, 0, 1, 0, 0, 1, 0])
toy_flag = np.array([0, 1, 1, 1, 0, 0, 0, 0, 1, 0])

toy_tp = ((toy_flag == 1) & (toy_y == 1)).sum()
toy_fp = ((toy_flag == 1) & (toy_y == 0)).sum()
toy_fn = ((toy_flag == 0) & (toy_y == 1)).sum()

toy_precision = toy_tp / (toy_tp + toy_fp)   # of the 4 flagged, how many really churn
toy_recall = toy_tp / (toy_tp + toy_fn)      # of the 3 churners, how many were flagged
toy_precision, toy_recall

In [ ]:
# WHY: if we send a discount to everyone we flag, precision tells us how much of that money goes to real churners.
p = tp / (tp + fp)
p

In [ ]:
# WHY: recall tells us how many churners we LOSE without ever contacting them (1 - recall of them).
#      Here accuracy looked fine, yet a large share of the churners slips through.
r = tp / (tp + fn)
r

In [ ]:
# Cross-check with scikit-learn.
from sklearn.metrics import precision_score, recall_score

precision_score(y_val, y_pred >= 0.5), recall_score(y_val, y_pred >= 0.5)

## 4.5 ROC curves

### TPR and FPR

Two rates, one per row of the confusion table:
- **TPR** (true positive rate) = TP / (TP + FN), the same thing as recall: share of churners we catch. Higher is better.
- **FPR** (false positive rate) = FP / (FP + TN): share of loyal customers we wrongly flag. Lower is better.

A ROC curve tracks both of them over **all** thresholds instead of committing to 0.5.

### In plain words: the ROC curve is the threshold dial, drawn

Each threshold gives one pair of numbers: **FPR** (what share of loyal customers we wrongly flag) and **TPR** (what share of churners we catch). Turning the dial from "flag nobody" down to "flag everybody" moves that pair from (0, 0) to (1, 1). The path it traces is the ROC curve.

How to read it: the **diagonal** is a coin flip (every extra churner caught costs the same share of false alarms). The **top-left corner** is perfect (all churners caught, no false alarms). A good model bows towards that corner.

The toy cell below shows three dial settings on made-up scores:

In [ ]:
toy_y     = np.array([0, 0, 1, 0, 0, 1, 0, 0, 1, 0])
toy_score = np.array([0.10, 0.40, 0.80, 0.60, 0.20, 0.35, 0.15, 0.30, 0.90, 0.05])

for thr in [0.7, 0.5, 0.3]:
    toy_flag = toy_score >= thr
    toy_tp = (toy_flag & (toy_y == 1)).sum()
    toy_fp = (toy_flag & (toy_y == 0)).sum()
    toy_tpr = toy_tp / (toy_y == 1).sum()     # share of churners caught
    toy_fpr = toy_fp / (toy_y == 0).sum()     # share of loyal customers wrongly flagged
    print('threshold %.1f   TPR %.2f   FPR %.2f' % (thr, toy_tpr, toy_fpr))

# WHY: lowering the threshold raises BOTH rates. A good model raises TPR much faster than FPR.

In [ ]:
tpr = tp / (tp + fn)
tpr

In [ ]:
fpr = fp / (fp + tn)
fpr

In [ ]:
# GOAL: compute the four counts at 101 thresholds: 0.00, 0.01, ..., 1.00.
scores = []

thresholds = np.linspace(0, 1, 101)

for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)

    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()

    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()

    # PYTHON: (t, tp, fp, fn, tn) is a TUPLE: one row of results; we collect the rows in a list.
    scores.append((t, tp, fp, fn, tn))

In [ ]:
# GOAL: list of tuples -> DataFrame, then add the two rates as new columns.
# PYTHON: pd.DataFrame(list_of_tuples, columns=[...]) names the columns in tuple order.
#         Column arithmetic (df.tp / (df.tp + df.fn)) runs on all 101 rows at once.
columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
df_scores = pd.DataFrame(scores, columns=columns)

df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)

# PYTHON: [::10] = every 10th row (slicing with a step).
df_scores[::10]

In [ ]:
# WHY: at threshold 0 everyone is flagged (TPR = FPR = 1); at threshold 1 nobody is (both 0).
#      A good model lets FPR fall much faster than TPR as the threshold rises.
plt.plot(df_scores.threshold, df_scores['tpr'], label='TPR')
plt.plot(df_scores.threshold, df_scores['fpr'], label='FPR')
plt.legend()

### Random model

The floor: scores drawn at random, with no information about the customer.

In [ ]:
# PYTHON: np.random.seed fixes the random generator so every run gives the same "random" numbers.
#         uniform(0, 1, size=n) draws n numbers evenly between 0 and 1.
np.random.seed(1)
y_rand = np.random.uniform(0, 1, size=len(y_val))

In [ ]:
# WHY: about 50% accuracy: a coin flip. (Note that it is WORSE than the dummy "nobody churns" model.)
((y_rand >= 0.5) == y_val).mean()

In [ ]:
# GOAL: wrap the 4.5 loop in a FUNCTION so we can run it for any set of scores (random, ideal, real model).
# PYTHON: def name(arguments): ... return value. Variables created inside exist only inside the function.
def tpr_fpr_dataframe(y_val, y_pred):
    scores = []

    thresholds = np.linspace(0, 1, 101)

    for t in thresholds:
        actual_positive = (y_val == 1)
        actual_negative = (y_val == 0)

        predict_positive = (y_pred >= t)
        predict_negative = (y_pred < t)

        tp = (predict_positive & actual_positive).sum()
        tn = (predict_negative & actual_negative).sum()

        fp = (predict_positive & actual_negative).sum()
        fn = (predict_negative & actual_positive).sum()

        scores.append((t, tp, fp, fn, tn))

    columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
    df_scores = pd.DataFrame(scores, columns=columns)

    df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
    df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)

    return df_scores

In [ ]:
# WHY: for random scores TPR and FPR fall together along the diagonal: flagging more people catches churners
#      and loyal customers in the same proportion. That diagonal is the line to beat.
df_rand = tpr_fpr_dataframe(y_val, y_rand)

plt.plot(df_rand.threshold, df_rand['tpr'], label='TPR')
plt.plot(df_rand.threshold, df_rand['fpr'], label='FPR')
plt.legend()

### Ideal model

The ceiling: a model that gives every churner a higher score than every non-churner.

In [ ]:
num_neg = (y_val == 0).sum()
num_pos = (y_val == 1).sum()
num_neg, num_pos

In [ ]:
# GOAL: build a perfectly ordered fake dataset: all non-churners first, then all churners.
# PYTHON: np.repeat([0, 1], [a, b]) -> a zeros followed by b ones.
y_ideal = np.repeat([0, 1], [num_neg, num_pos])

# Scores rising from 0 to 1 in the same order, so every churner outranks every non-churner.
y_ideal_pred = np.linspace(0, 1, len(y_val))

# WHY: the right threshold is the share of non-churners (the lecture types 0.726). Every score below it belongs
#      to a non-churner, every score above it to a churner, so accuracy is 1.0.
t_ideal = 1 - y_val.mean()
t_ideal, accuracy_score(y_ideal, y_ideal_pred >= t_ideal)

In [ ]:
df_ideal = tpr_fpr_dataframe(y_ideal, y_ideal_pred)
df_ideal[::10]

In [ ]:
# WHY: TPR stays at 1 until the threshold passes the last non-churner, while FPR drops straight to 0.
plt.plot(df_ideal.threshold, df_ideal['tpr'], label='TPR')
plt.plot(df_ideal.threshold, df_ideal['fpr'], label='FPR')
plt.legend()

### Putting everything together

In [ ]:
# GOAL: our model against the ideal one on the same axes: how much room is left?
plt.plot(df_scores.threshold, df_scores['tpr'], label='TPR', color='black')
plt.plot(df_scores.threshold, df_scores['fpr'], label='FPR', color='blue')

plt.plot(df_ideal.threshold, df_ideal['tpr'], label='TPR ideal')
plt.plot(df_ideal.threshold, df_ideal['fpr'], label='FPR ideal')

plt.legend()

In [ ]:
# GOAL: the ROC curve itself: drop the threshold axis and plot TPR against FPR.
# WHY: each point is one threshold. Random = the diagonal, ideal = the top-left corner (FPR 0, TPR 1).
#      The closer our curve bends towards that corner, the better the model at EVERY threshold.
# PYTHON: plt.figure(figsize=(5, 5)) makes a square figure, so the diagonal really is at 45 degrees.
plt.figure(figsize=(5, 5))

plt.plot(df_scores.fpr, df_scores.tpr, label='Model')
plt.plot([0, 1], [0, 1], label='Random', linestyle='--')

plt.xlabel('FPR')
plt.ylabel('TPR')

plt.legend()

In [ ]:
# GOAL: the production way: scikit-learn computes the curve for us.
# PYTHON: roc_curve returns THREE arrays, unpacked into three names in one line (tuple unpacking).
#         It picks the thresholds itself (every distinct score), so the curve is finer than our 101 points.
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_val, y_pred)

In [ ]:
plt.figure(figsize=(5, 5))

plt.plot(fpr, tpr, label='Model')
plt.plot([0, 1], [0, 1], label='Random', linestyle='--')

plt.xlabel('FPR')
plt.ylabel('TPR')

plt.legend()

## 4.6 ROC AUC

The area under the ROC curve squeezes the whole curve into one number: 0.5 = random, 1.0 = ideal.
It does not depend on any threshold, and it is not fooled by class imbalance the way accuracy is.

### In plain words: what AUC measures

AUC answers one question: **pick one churner and one loyal customer at random; how often does the model give the churner the higher score?** 1.0 = always, 0.5 = a coin flip, below 0.5 = the model ranks backwards (HW4 Q1 tells you to negate such a feature).

Two consequences. AUC looks only at the *order* of the scores, so no threshold is involved. And it does not care how many churners there are, so class imbalance cannot flatter it the way it flatters accuracy.

That is also why `roc_auc_score` wants **scores** (probabilities), never 0/1 decisions: turning scores into decisions throws the ordering away. Toy check: count the winning pairs by hand and compare with scikit-learn.

In [ ]:
from sklearn.metrics import roc_auc_score

toy_y     = np.array([0, 0, 1, 0, 0, 1, 0, 0, 1, 0])
toy_score = np.array([0.10, 0.40, 0.80, 0.60, 0.20, 0.35, 0.15, 0.30, 0.90, 0.05])

toy_pos = toy_score[toy_y == 1]     # scores of the churners
toy_neg = toy_score[toy_y == 0]     # scores of the loyal customers

# every (churner, loyal) pair: did the churner get the higher score?
toy_wins = sum(p > n for p in toy_pos for n in toy_neg)
toy_pairs = len(toy_pos) * len(toy_neg)

# the hand count and scikit-learn's AUC should be the same number
toy_wins, toy_pairs, toy_wins / toy_pairs, roc_auc_score(toy_y, toy_score)

In [ ]:
# PYTHON: auc(x, y) is a general "area under a curve" function (trapezoids). x first: FPR, then TPR.
from sklearn.metrics import auc

auc(fpr, tpr)

In [ ]:
# WHY: our hand-made 101-point curve gives almost the same area: the curve, not the tool, decides the number.
auc(df_scores.fpr, df_scores.tpr)

In [ ]:
auc(df_ideal.fpr, df_ideal.tpr)    # the ceiling: 1.0

In [ ]:
# GOAL: the one-liner you will actually use: AUC straight from labels and scores.
# WHY: pass PROBABILITIES (y_pred), not 0/1 decisions. With decisions you get the area of a single-threshold
#      "curve", which is a different (and worse) number.
from sklearn.metrics import roc_auc_score

roc_auc_score(y_val, y_pred)

In [ ]:
# GOAL: the INTERPRETATION of AUC: the probability that a random churner gets a higher score than
#       a random non-churner. Check it by sampling pairs.
# PYTHON: boolean masks select scores: y_pred[y_val == 0] keeps only the scores of non-churners.
neg = y_pred[y_val == 0]
pos = y_pred[y_val == 1]

In [ ]:
# Version 1: a plain Python loop, one random pair at a time (slow but easy to read).
import random

n = 100000
success = 0

for i in range(n):
    # PYTHON: random.randint(a, b) includes BOTH ends, hence len(...) - 1.
    pos_ind = random.randint(0, len(pos) - 1)
    neg_ind = random.randint(0, len(neg) - 1)

    if pos[pos_ind] > neg[neg_ind]:
        success = success + 1

success / n

In [ ]:
# Version 2: the same experiment vectorised with NumPy: 50,000 pairs in one line, no loop.
# PYTHON: np.random.randint(a, b, size=n) EXCLUDES b, so len(pos) is the right upper bound here.
n = 50000

np.random.seed(1)
pos_ind = np.random.randint(0, len(pos), size=n)
neg_ind = np.random.randint(0, len(neg), size=n)

# WHY: both versions land next to roc_auc_score above: that is what AUC means.
(pos[pos_ind] > neg[neg_ind]).mean()

## 4.7 Cross-validation

One validation split gives one number, and a different split would give a slightly different one.
**K-fold cross-validation** trains and scores K times on K different validation folds of `df_full_train`,
so we get a mean *and* a spread (std). The test set still stays untouched until the very end.

### In plain words: why repeat the test?

One train/validation split gives one number, and that number depends on *which* customers happened to land in validation. A lucky split flatters the model; an unlucky one hides a good one. K-fold lets every customer take a turn in validation:

```text
5 folds of df_full_train   (V = validation part, T = training part)
fold 1:  V T T T T
fold 2:  T V T T T
fold 3:  T T V T T
fold 4:  T T T V T
fold 5:  T T T T V
```

Each fold trains a fresh model on its four T parts and scores it on its V part: five scores. Read them like this: the **mean** is your estimate of how good the model is, and the **std** is how much that estimate wobbles. If two settings of `C` differ by less than the std, you cannot really tell them apart.

Every fold starts from scratch, including a **new** `DictVectorizer` fitted on that fold's training part only and then used to transform its validation part: the usual rule, *fit on the training part, transform everything else*. The test set stays out of all this until the very end.

In [ ]:
# GOAL: package "dataframe in -> fitted (dv, model) out" as a function, so each fold is one call.
# WHY: the DictVectorizer is created and fitted INSIDE the function: every fold learns its layout from
#      its own training part only. A dv fitted once outside would leak the validation fold.
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    # PYTHON: C=C passes the function's argument on to the model (smaller C = stronger regularisation).
    # NOTE: the CV loop below prints a wall of ConvergenceWarnings (unscaled totalcharges, see 4.1).
    #       They don't change the conclusions. HW4's solver='liblinear' doesn't have this problem.
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train, y_train)

    # PYTHON: returning two values returns a tuple; the caller unpacks it: dv, model = train(...)
    return dv, model

In [ ]:
dv, model = train(df_train, y_train, C=0.001)

In [ ]:
# GOAL: the matching "dataframe in -> churn probabilities out" function (transform only, never fit).
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [ ]:
y_pred = predict(df_val, dv, model)

In [ ]:
# GOAL: 5-fold CV for several values of C.
# PYTHON: KFold(...).split(df) yields (train_idx, val_idx) pairs of ROW POSITIONS, one pair per fold.
#         .iloc[...] selects rows by position (not by index label).
#         shuffle=True + random_state=1 makes the folds random but reproducible (HW4 uses exactly these).
from sklearn.model_selection import KFold

n_splits = 5

for C in [0.001, 0.01, 0.1, 0.5, 1, 5, 10]:
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

    scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_idx]
        df_val = df_full_train.iloc[val_idx]

        # WHY: df_full_train still has 'churn', so we take y from each fold here. The feature lists
        #      (categorical + numerical) do not contain 'churn', so the target cannot leak into X.
        y_train = df_train.churn.values
        y_val = df_val.churn.values

        dv, model = train(df_train, y_train, C=C)
        y_pred = predict(df_val, dv, model)

        # The lecture calls this variable 'auc', which overwrites sklearn's auc() function from 4.6.
        score = roc_auc_score(y_val, y_pred)
        scores.append(score)

    # WHY: read the MEAN to compare models and the STD to judge whether a difference is real:
    #      two C values whose means differ by less than the std are, in practice, a tie.
    print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

In [ ]:
scores    # the five fold AUCs of the last C in the loop

In [ ]:
# GOAL: the final model. Pick C from CV, retrain on ALL of df_full_train, score ONCE on the untouched test set.
# WHY: if the test AUC is close to the CV mean, the CV estimate was honest and the model generalises.
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

score = roc_auc_score(y_test, y_pred)
score

## 4.8 Summary

| Metric | Answers | Watch out |
|---|---|---|
| Accuracy | share of correct decisions | flattered by class imbalance: always compare with the dummy model |
| Confusion table | which kind of mistake we make | depends on the threshold |
| Precision | of those we flag, how many are right | ignores the churners we miss |
| Recall (= TPR) | of the real positives, how many we catch | ignores false alarms |
| ROC curve / AUC | ranking quality over all thresholds | pass probabilities, not 0/1 decisions |
| K-fold CV | mean **and** spread of a metric | fit every preprocessing step inside the fold |

## 4.9 Explore more

- Precision and recall of the dummy "nobody churns" classifier (what happens to precision?)
- F1 = 2 · P · R / (P + R) at every threshold: where does it peak?
- Plot precision against recall for all thresholds: the precision-recall curve, and the area under it